# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`06_orchard_estimate.ipynb`**

Three questions, in the client's words.

```
1  What is the configuration that takes the most fruit the robot can reach?
2  How much does it take from one tree?
3  How long does that take?
```

**The denominator is fruit the robot can reach** — with the base free to move along the row and
the mast free to rise. Fruit hidden from the detector and fruit outside that envelope are not
inputs to this system; a person picks those. That is a definition of scope, not a claim about
what a person can or cannot do, and no comparison against human picking is made anywhere here.

The planning itself is `src/planner.py`. This notebook chooses configurations, runs them over the
held-out trees, and aggregates — it holds no planning logic of its own. That separation is not
tidiness: the same logic was written into three notebooks before it was written into a module,
and the three drifted until their figures could not be put in the same table.

**Two settings were re-chosen for this client.** Stops per tree were three, which maximises
premium fruit per hour — the right objective for a machine skimming an unbounded orchard, and the
wrong one for a grower who has to see a finite block picked, since fruit the robot skips is handed
to a person rather than saved. Twenty stops plus a sweep stage reach everything the arm can touch.
The selection threshold was 0.9, which left roughly a third of the reachable fruit alone to avoid
disturbing neighbours; at zero the tree yields far more premium fruit for a few more on the ground.

The estimate treats thirty held-out trees as a sample. Seeds are not involved: the plan is
deterministic, and what is projected is the plan and its expected value.


In [1]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd

# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)
SRC, DATA, MODELS = ROOT/"src", ROOT/"data", ROOT/"models"
OUT = ROOT/"runs"/"orchard"; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

import environment as E
E.load(ROOT, trees="trees_measured_pose.csv", dynamics=True)

import planner as PL
info = PL.load(ROOT)

TREES = list(range(20, 50))      # held out from every stage of training
# Read from 04, not copied by hand. The gap has to come from the same operating point the
# estimate describes: measured at a 0.9 threshold it only ever saw the picks the outcome model
# was already confident about, and applying that to a plan which attempts everything would
# understate the correction.
_gap_file = ROOT/"runs"/"fidelity"/"fidelity_gap.json"
if _gap_file.exists():
    _g = json.loads(_gap_file.read_text())
    FIDELITY_GAP = -_g["gap"]
    print(f"fidelity gap {FIDELITY_GAP:.3f} from {_gap_file.name} "
          f"({_g['n']:,} picks, picker {_g['picker']}, {_g['stops']} stops, "
          f"threshold {_g['threshold']})")
else:
    FIDELITY_GAP = 0.134
    print(f"fidelity gap {FIDELITY_GAP:.3f} -- 04_fidelity has not been run at this "
          f"configuration, so this is the earlier figure and understates the correction")
SHIFT_SECONDS = 3600.0
BETWEEN = E.TREE_SPACING/E.TRAVEL

pd.set_option("display.width", 220)
print(f"canopy {len(E.T):,} fruit / {E.T.tree.nunique()} trees")
print(f"planner {info['planner']}   policy {info['policy']}   scaling {info['scaling']}")
print(f"configuration: arm {PL.HALF_X} x {PL.LIFT} m, {PL.K_STATIONS} stops, "
      f"threshold {PL.THRESHOLD}, sweep {'on' if PL.SWEEP else 'off'}")
print(f"pick cycle {E.PICK_SECONDS:.0f} s   between trees {BETWEEN:.0f} s   "
      f"dynamics {'on' if E.DYN else 'off'}")




c:\python\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fidelity gap 0.058 from fidelity_gap.json (1,820 picks, picker rule, 20 stops, threshold 0.0)
canopy 6,000 fruit / 50 trees
planner station_planner_k20.pt   policy pick_policy.pt   scaling recomputed
configuration: arm 0.2 x 0.6 m, 20 stops, threshold 0.0, sweep on
pick cycle 14 s   between trees 3 s   dynamics on


## 1. Plan every held-out tree, under four configurations

The shipping configuration and the three it is measured against. Each is the same call with
different keywords, which is the point of having the module.

```
chosen         planner + sweep, rate rule, threshold 0     what the report describes
planner only   the sweep turned off                        what the sweep adds
coverage       coverage search instead of the planner      whether the network earns its place
earlier        three stops, threshold 0.9, no sweep        the operating point this replaces
```


In [2]:
CONFIGS = (
    ("chosen",       dict()),
    ("planner only", dict(sweep=False)),
    ("coverage",     dict(chooser="greedy")),
    ("earlier",      dict(k=3, threshold=0.9, chooser="greedy", sweep=False)),
)

rows, t0 = [], time.time()
for tag, kw in CONFIGS:
    S = PL.plan_summary(TREES, **kw)
    S["config"] = tag
    rows.append(S)
    print(f"  {tag:<14} {len(S)} trees   {(time.time()-t0)/60:5.1f} min")

TREE = pd.concat(rows, ignore_index=True)
TREE.to_csv(OUT/"per_tree.csv", index=False)
print(f"\n{len(TREE)} rows -> {OUT/'per_tree.csv'}")


  chosen         30 trees     0.3 min
  planner only   30 trees     0.5 min
  coverage       30 trees     0.7 min
  earlier        30 trees     0.9 min

120 rows -> c:\aipick\git\runs\orchard\per_tree.csv


In [3]:
G = TREE.groupby("config", sort=False).agg(
    stops=("stops", "mean"), stage1=("stage1_stops", "mean"), sweep=("sweep_stops", "mean"),
    in_plan=("in_plan", "mean"), reach=("robot_reach", "mean"),
    attempts=("attempts", "mean"), premium=("exp_success", "mean"),
    utility=("exp_utility", "mean"), seconds=("seconds", "mean")).round(2)
G["cover_pct"] = (G.in_plan/G.reach*100).round(1)
G["s_per_premium"] = (G.seconds/G.premium.clip(lower=0.01)).round(1)

print(f"per tree, {len(TREES)} held-out trees\n")
print(G[["stops", "stage1", "sweep", "in_plan", "cover_pct", "attempts",
         "premium", "utility", "seconds", "s_per_premium"]].to_string())

CH = TREE[TREE.config == "chosen"]
PO = TREE[TREE.config == "planner only"]
CO = TREE[TREE.config == "coverage"]

d_p = CH.exp_success.mean() - PO.exp_success.mean()
d_s = CH.seconds.mean() - PO.seconds.mean()
print(f"\n\nwhat the sweep adds\n")
print(f"  {CH.sweep_stops.mean():.1f} stops, {CH.attempts.mean() - PO.attempts.mean():+.1f} "
      f"attempts, {d_p:+.2f} premium, {d_s:+.0f} s")
if d_p > 0.01:
    print(f"  a sweep fruit costs {d_s/d_p:.0f} s against "
          f"{PO.seconds.mean()/PO.exp_success.mean():.0f} s in the first pass")

print(f"\n\nplanner against coverage search, both with the sweep\n")
print(f"  {'':<14}{'stage 1':>9}{'sweep':>8}{'in plan':>9}{'premium':>9}")
for lbl, D in (("planner", CH), ("coverage", CO)):
    print(f"  {lbl:<14}{D.stage1_stops.mean():>9.1f}{D.sweep_stops.mean():>8.1f}"
          f"{D.in_plan.mean():>9.1f}{D.exp_success.mean():>9.1f}")
print("\n  They finish in the same place. The planner has no stopping rule and spends all of")
print("  its stops; coverage search halts at its first zero-gain step and leans on the sweep")
print("  to make up the difference. Same result, fewer sweep stops -- that is what the network")
print("  buys here, and it is worth saying plainly rather than dressing up.")


per tree, 30 held-out trees

              stops  stage1  sweep  in_plan  cover_pct  attempts  premium  utility  seconds  s_per_premium
config                                                                                                    
chosen        21.27    20.0   1.27    62.57      100.0     62.57    49.92    45.83  1227.18           24.6
planner only  20.00    20.0   0.00    61.23       97.9     61.23    49.24    45.40  1203.58           24.4
coverage      21.33    18.3   3.03    62.57      100.0     62.57    49.92    45.83  1227.88           24.6
earlier        3.00     3.0   0.00    19.57       31.3     14.00    13.75    13.70   250.26           18.2


what the sweep adds

  1.3 stops, +1.3 attempts, +0.68 premium, +24 s
  a sweep fruit costs 35 s against 24 s in the first pass


planner against coverage search, both with the sweep

                  stage 1   sweep  in plan  premium
  planner            20.0     1.3     62.6     49.9
  coverage           18.3     3.0     6

## 2. Question 2 — how much does one tree give?

Four denominators, and confusing them is how "the robot takes 8% of a tree" gets written down.
Only the last step is a decision the farm can change; the two before it are what the machine and
the detector can do.


In [4]:
det = E.T[E.T.tree.isin(TREES)].groupby("tree").vis_any.sum().mean() \
    if "vis_any" in E.T else float("nan")

print("per tree, averaged over 30 held-out trees\n")
print(f"  fruit on the tree                {CH.on_tree.mean():7.1f}")
print(f"  detected                         {det:7.1f}   "
      f"{det/CH.on_tree.mean()*100:4.0f}% of the tree   (foliage hides the rest)")
print(f"  the robot can reach              {CH.robot_reach.mean():7.1f}   "
      f"{CH.robot_reach.mean()/det*100:4.0f}% of detected   (base and mast at any position)")
print(f"  put in the plan                  {CH.in_plan.mean():7.1f}   "
      f"{CH.in_plan.mean()/CH.robot_reach.mean()*100:4.0f}% of reachable")
print(f"  attempted                        {CH.attempts.mean():7.1f}   "
      f"threshold {PL.THRESHOLD}")
print(f"  expected premium fruit           {CH.exp_success.mean():7.1f}   "
      f"{CH.exp_success.mean()/CH.attempts.mean()*100:4.0f}% of attempts succeed")
print(f"  left for a person, within reach  {CH.left_reachable.mean():7.1f}")

EA = TREE[TREE.config == "earlier"]
cmp = pd.DataFrame({
    "3 stops, threshold 0.9": [EA.in_plan.mean(), EA.attempts.mean(), EA.exp_success.mean(),
                               EA.exp_utility.mean(), EA.seconds.mean()],
    "20 stops + sweep, threshold 0": [CH.in_plan.mean(), CH.attempts.mean(),
                                      CH.exp_success.mean(), CH.exp_utility.mean(),
                                      CH.seconds.mean()],
}, index=["in the plan", "attempted", "expected premium", "expected utility",
          "seconds"]).round(2)
cmp["change"] = (cmp.iloc[:, 1] - cmp.iloc[:, 0]).round(2)
print("\n\nagainst the operating point it replaces\n")
print(cmp.to_string())

print("\n\ndistribution across trees, chosen configuration\n")
print(CH[["robot_reach", "in_plan", "attempts", "exp_success", "seconds"]].describe(
    percentiles=[.25, .5, .75]).round(2).to_string())


per tree, averaged over 30 held-out trees

  fruit on the tree                  120.0
  detected                            94.3     79% of the tree   (foliage hides the rest)
  the robot can reach                 62.6     66% of detected   (base and mast at any position)
  put in the plan                     62.6    100% of reachable
  attempted                           62.6   threshold 0.0
  expected premium fruit              49.9     80% of attempts succeed
  left for a person, within reach      0.0


against the operating point it replaces

                  3 stops, threshold 0.9  20 stops + sweep, threshold 0  change
in the plan                        19.57                          62.57   43.00
attempted                          14.00                          62.57   48.57
expected premium                   13.75                          49.92   36.17
expected utility                   13.70                          45.83   32.13
seconds                           250.26       

## 3. Question 3 — how long?

The cycle is dominated by picking, not by moving, which is why adding stops cost so little.

Two things this figure does not include. **The pick cycle is a literature-based constant, not a
measurement from a machine** — halving it is worth more than every planning decision in this
project combined, so it is the number to interrogate first. And **a fruit is attempted once**:
there is no retry after a failed grasp, which understates both the time and the yield.

Settling is not missing from it. A separate sweep measured what happens when a neighbour is
taken and found that reading the approach axis two seconds into the park recovers the success
rate at no cost, because the park happens regardless. The planner never has to wait.


In [5]:
picks = CH.attempts.mean()
secs = CH.seconds.mean()
travel = CH.travel.mean()
print(f"one tree, {PL.K_STATIONS} stops plus sweep, threshold {PL.THRESHOLD}\n")
print(f"  attempts                {picks:8.1f}")
print(f"  picking                 {picks*E.PICK_SECONDS:8.0f} s   "
      f"{picks*E.PICK_SECONDS/secs*100:4.0f}%")
print(f"  moving between stops    {travel:8.0f} s   {travel/secs*100:4.0f}%")
print(f"  total on the tree       {secs:8.0f} s   = {secs/60:.1f} min")
print(f"  driving to the next     {BETWEEN:8.0f} s")
print(f"  tree to tree            {secs + BETWEEN:8.0f} s   "
      f"-> {SHIFT_SECONDS/(secs+BETWEEN):.1f} trees in an hour")
print(f"\n  {secs/picks:.1f} s per attempt, {secs/CH.exp_success.mean():.1f} s per premium fruit")


one tree, 20 stops plus sweep, threshold 0.0

  attempts                    62.6
  picking                      876 s     71%
  moving between stops         351 s     29%
  total on the tree           1227 s   = 20.5 min
  driving to the next            3 s
  tree to tree                1230 s   -> 2.9 trees in an hour

  19.6 s per attempt, 24.6 s per premium fruit


## 4. Question 1, at orchard scale

Thirty trees are a sample. Trees are resampled with replacement and the totals recomputed, which
carries tree-to-tree variation into the interval without assuming the totals are normal.

This is a projection from a synthetic canopy, not a forecast for a real block. What it supports
is planning arithmetic — how many shifts, how much fruit, how many people.


In [6]:
N_ORCHARD = 500
N_BOOT = 20000
RNG = np.random.default_rng(0)

COLS = ["attempts", "exp_success", "exp_utility", "seconds"]
A = CH[COLS].to_numpy(float)
idx = RNG.integers(0, len(A), (N_BOOT, len(A)))
boot = A[idx].mean(axis=1)*N_ORCHARD
point = A.mean(0)*N_ORCHARD
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

O = pd.DataFrame({"estimate": point, "lo": lo, "hi": hi}, index=COLS)
O.loc["seconds"] += BETWEEN*N_ORCHARD
O.loc["hours"] = O.loc["seconds"]/3600.0
O = O.drop(index="seconds")

print(f"orchard of {N_ORCHARD} trees\n")
print(O.round(1).to_string())
print(f"\n  {O.loc['hours','estimate']:.0f} robot hours "
      f"[{O.loc['hours','lo']:.0f}, {O.loc['hours','hi']:.0f}]"
      f"  = {O.loc['hours','estimate']/8:.0f} eight-hour days of one machine")
print(f"  {O.loc['exp_success','estimate']:,.0f} premium fruit "
      f"[{O.loc['exp_success','lo']:,.0f}, {O.loc['exp_success','hi']:,.0f}]")

total = CH.on_tree.mean()*N_ORCHARD
print(f"\n  of {total:,.0f} fruit in the block the robot attempts "
      f"{CH.attempts.mean()*N_ORCHARD:,.0f} ({CH.attempts.mean()/CH.on_tree.mean()*100:.0f}%)")
print(f"  the rest: {(CH.on_tree.mean()-det)*N_ORCHARD:,.0f} never detected, "
      f"{(det-CH.robot_reach.mean())*N_ORCHARD:,.0f} out of the robot's reach, "
      f"{CH.left_reachable.mean()*N_ORCHARD:,.0f} reachable but not planned")


orchard of 500 trees

             estimate       lo       hi
attempts      31283.3  30500.0  32083.7
exp_success   24960.7  24223.0  25668.5
exp_utility   22915.8  22136.8  23633.9
hours           170.9    165.9    175.9

  171 robot hours [166, 176]  = 21 eight-hour days of one machine
  24,961 premium fruit [24,223, 25,668]

  of 60,000 fruit in the block the robot attempts 31,283 (52%)
  the rest: 12,867 never detected, 15,850 out of the robot's reach, 0 reachable but not planned


## 5. By grade, before the harvest starts

The requirement the client stated most directly. The outcome model gives a distribution over six
classes for every fruit it plans, so the breakdown falls out of the plan.

`SUCCESS` is a premium fruit with its stem attached. `NEIGHBOR_KNOCKED` is fruit on the ground —
it exists, and in practice it is collected, inspected and sold for processing, but it is not
premium. `DEFECT` is a torn stem, which is what the grade standard penalises.


In [7]:
# Class shares, from the plans already built above. Re-planning every tree here was both slow
# and noisy; the posteriors are re-derived once per tree from the same deterministic plan.
col = {c: i for i, c in enumerate(E.CLASSES)}
tot = {c: 0.0 for c in E.CLASSES}
for tid in TREES:
    case = PL.tree_case(tid)
    p = PL.plan_tree(case)
    st = E.TreeState(case["geom"])
    ids = case["geom"].ids
    for f in p.fruit:
        j = int(np.where(ids == f)[0][0])
        pr = st.probabilities(j)
        for c, i in col.items():
            tot[c] += float(pr[i])
        st.remove(j)

n = sum(tot.values())
scale = N_ORCHARD/len(TREES)
LABEL = {"SUCCESS": "premium, stem intact",
         "NEIGHBOR_KNOCKED": "knocked to the ground",
         "DEFECT": "stem torn", "STEM_PULL": "stem pulled",
         "STALK_SNAP": "stalk snapped", "SPUR_BREAK": "spur broken",
         "GRASP_FAILED": "not detached, grasp failed",
         "APPROACH_BLOCKED": "not attempted, no corridor",
         "NO_DETACH": "not detached"}

print(f"{N_ORCHARD} trees, {n*scale:,.0f} attempts\n")
for c, v in sorted(tot.items(), key=lambda kv: -kv[1]):
    if v*scale < 1:
        continue
    print(f"  {LABEL.get(c, c):<30} {v*scale:>10,.0f}   {v/n*100:5.1f}%")
print(f"\n  expected utility {CH.exp_utility.mean()*N_ORCHARD:,.0f} grade units "
      f"(premium +1, knocked -0.5, torn stem -0.3)")
print("\n  Knocked fruit is not lost. It is collected off the ground, inspected and sold for")
print("  processing -- a grade change rather than a write-off, which is why the utility table")
print("  charges it half a premium fruit and not a whole one.")

GRADE = pd.DataFrame([(LABEL.get(c, c), v*scale, v/n*100) for c, v in
                      sorted(tot.items(), key=lambda kv: -kv[1]) if v*scale >= 1],
                     columns=["outcome", "count", "percent"])
GRADE.to_csv(OUT/"grade_mix.csv", index=False)
print(f"\nwritten {OUT/'grade_mix.csv'}")


500 trees, 31,283 attempts

  premium, stem intact               24,961    79.8%
  knocked to the ground               4,008    12.8%
  not attempted, no corridor          1,976     6.3%
  not detached, grasp failed            191     0.6%
  stem torn                             137     0.4%
  not detached                           10     0.0%

  expected utility 22,916 grade units (premium +1, knocked -0.5, torn stem -0.3)

  Knocked fruit is not lost. It is collected off the ground, inspected and sold for
  processing -- a grade change rather than a write-off, which is why the utility table
  charges it half a premium fruit and not a whole one.

written c:\aipick\git\runs\orchard\grade_mix.csv


## 6. Labour

The robot's side is measured; the human side is not. This project makes no comparison against
human picking, so the rate below is an **input** to be replaced with the farm's own records. What
the estimate provides is the split.


In [8]:
HUMAN_FRUIT_PER_HOUR = 300      # INPUT -- replace with the farm's own figure
WORKDAY_HOURS = 8

robot_h = float(O.loc["hours", "estimate"])
robot_fruit = CH.attempts.mean()*N_ORCHARD
left_fruit = (CH.on_tree.mean() - CH.attempts.mean())*N_ORCHARD

print(f"orchard of {N_ORCHARD} trees\n")
print(f"  fruit in the block            {CH.on_tree.mean()*N_ORCHARD:>11,.0f}")
print(f"  attempted by the robot        {robot_fruit:>11,.0f}   "
      f"{robot_fruit/(CH.on_tree.mean()*N_ORCHARD)*100:.0f}%")
print(f"  left for people               {left_fruit:>11,.0f}")
print(f"\n  robot time                    {robot_h:>11,.0f} h   "
      f"= {robot_h/WORKDAY_HOURS:.0f} days of one machine")
print(f"\n  the line above is measured; the ones below follow from an assumed rate\n")
for rate in (200, 300, 400, 500):
    h = left_fruit/rate
    print(f"    at {rate:>4}/h   {h:>9,.0f} h   {h/WORKDAY_HOURS:>6.0f} person-days")
print(f"\n  Robot hours are machine time; person-hours are people who have to be hired. The")
print(f"  value is that nobody is called for the {robot_fruit:,.0f} the robot attempts, not")
print(f"  that the machine is faster.")


orchard of 500 trees

  fruit in the block                 60,000
  attempted by the robot             31,283   52%
  left for people                    28,717

  robot time                            171 h   = 21 days of one machine

  the line above is measured; the ones below follow from an assumed rate

    at  200/h         144 h       18 person-days
    at  300/h          96 h       12 person-days
    at  400/h          72 h        9 person-days
    at  500/h          57 h        7 person-days

  Robot hours are machine time; person-hours are people who have to be hired. The
  value is that nobody is called for the 31,283 the robot attempts, not
  that the machine is faster.


## 7. What the physics said about the same plans

Everything above is the outcome model's expectation. `04_fidelity` re-ran the planner's own
choices against MuJoCo over 829 picks and found the realised success rate 0.134 below the
predicted one.

Report both. The predicted figure is what the planner believes; the corrected one is what the
physics delivered on the same plans; neither measures an orchard. The correction is also a rough
instrument — the gap was measured at a different operating point, and there is no reason it
should be constant across them.


In [9]:
attempts = CH.attempts.mean()*N_ORCHARD
pred = CH.exp_success.sum()/CH.attempts.sum()
real = max(pred - FIDELITY_GAP, 0.0)

print(f"{N_ORCHARD} trees, {attempts:,.0f} attempts\n")
print(f"  predicted success rate        {pred:6.3f}   {attempts*pred:>10,.0f} premium")
print(f"  corrected for the surrogate   {real:6.3f}   {attempts*real:>10,.0f} premium")
print(f"  difference                             {(pred-real)*attempts:>17,.0f}")

print(f"\n  Four measurements, and they are not a single sequence of improvements:")
print(f"    -0.344   the first canopy, before the stalk-direction defect was found")
print(f"    -0.172   the same plans once the canopy was resampled from the detections")
print(f"    -0.134   with the pose measured, at three stops and a 0.9 threshold")
print(f"    -{FIDELITY_GAP:.3f}   here, at twenty stops with no threshold")
print(f"\n  The first two steps were fixes. The last one is not a better model -- it is a wider")
print(f"  plan, and the gap depends on where it is measured. A selection threshold scores the")
print(f"  model only where it is confident, which is where its optimism is largest: at 0.9 the")
print(f"  gap is -0.120. Attempting everything brings in the low-probability picks, where the")
print(f"  model under-predicts by +0.24, and the two errors partly cancel.")
print(f"\n  What remains tracks a canopy denser than the detections. That was deliberate --")
print(f"  matching them produced trees with no touching fruit, which deletes the problem the")
print(f"  planner exists to solve.")

summary = dict(
    config=dict(stops=PL.K_STATIONS, sweep=PL.SWEEP, threshold=PL.THRESHOLD,
                arm=[PL.HALF_X, PL.LIFT], chooser="planner", picker="rule",
                pick_seconds=float(E.PICK_SECONDS)),
    per_tree=dict(on_tree=float(CH.on_tree.mean()), detected=float(det),
                  robot_reach=float(CH.robot_reach.mean()),
                  in_plan=float(CH.in_plan.mean()), attempted=float(CH.attempts.mean()),
                  premium=float(CH.exp_success.mean()),
                  utility=float(CH.exp_utility.mean()),
                  stops=float(CH.stops.mean()), sweep_stops=float(CH.sweep_stops.mean()),
                  seconds=float(CH.seconds.mean())),
    orchard=dict(trees=N_ORCHARD, attempts=float(attempts),
                 premium_predicted=float(attempts*pred),
                 premium_corrected=float(attempts*real),
                 robot_hours=float(robot_h), left_for_people=float(left_fruit)),
    human_rate_input=HUMAN_FRUIT_PER_HOUR,
    fidelity=dict(gap=FIDELITY_GAP, measured_at="20 stops, threshold 0, rate rule",
                  note="not comparable to gaps measured with a selection threshold"))
(OUT/"orchard_estimate.json").write_text(json.dumps(summary, indent=1))
print(f"\nwritten {OUT/'orchard_estimate.json'}")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<26} {f.stat().st_size/1024:8.1f} KB")

500 trees, 31,283 attempts

  predicted success rate         0.798       24,961 premium
  corrected for the surrogate    0.740       23,148 premium
  difference                                         1,813

  Four measurements, and they are not a single sequence of improvements:
    -0.344   the first canopy, before the stalk-direction defect was found
    -0.172   the same plans once the canopy was resampled from the detections
    -0.134   with the pose measured, at three stops and a 0.9 threshold
    -0.058   here, at twenty stops with no threshold

  The first two steps were fixes. The last one is not a better model -- it is a wider
  plan, and the gap depends on where it is measured. A selection threshold scores the
  model only where it is confident, which is where its optimism is largest: at 0.9 the
  gap is -0.120. Attempting everything brings in the low-probability picks, where the
  model under-predicts by +0.24, and the two errors partly cancel.

  What remains tracks a can

### The three answers

```
1  twenty stops from the planner, a sweep stage for what they miss, no threshold
2  section 2 -- attempts and expected premium per tree, with the spread across trees
3  section 3 -- minutes per tree and trees per hour, with the pick cycle named as
   the assumption it is
```

`N_ORCHARD` and `HUMAN_FRUIT_PER_HOUR` are the two numbers to change for a particular block.
Everything about how the plan is made lives in `src/planner.py`.
